# Infeasible? Ask the solver for an IIS
[![iis_unit_commitment.ipynb](https://img.shields.io/badge/github-%23121011.svg?logo=github)](https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/solver-features/iis_unit_commitment.ipynb) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/solver-features/iis_unit_commitment.ipynb) [![Open In Deepnote](https://deepnote.com/buttons/launch-in-deepnote-small.svg)](https://deepnote.com/launch?url=https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/solver-features/iis_unit_commitment.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/solver-features/iis_unit_commitment.ipynb) [![Open In Gradient](https://assets.paperspace.io/img/gradient-badge.svg)](https://console.paperspace.com/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/solver-features/iis_unit_commitment.ipynb) [![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/ampl/colab.ampl.com/blob/master/authors/glebbelov/solver-features/iis_unit_commitment.ipynb) [![Powered by AMPL](https://h.ampl.com/https://github.com/ampl/colab.ampl.com/blob/master/authors/glebbelov/solver-features/iis_unit_commitment.ipynb)](https://ampl.com)

Description: Find out why a unit commitment model is infeasible: ask an MP solver for an irreducible infeasible subsystem (IIS) and read it through AMPL suffixes in amplpy

Tags: mp, iis, infeasibility, suffixes, unit-commitment, energy, milp, amplpy, highs, gurobi, xpress

<!--
Domain: Energy
Subdomain: Unit Commitment
Model type: MILP
Level: Lecture
Algorithm: Debugging Infeasibility
Subject: Irreducible Infeasible Subsystem
-->

Notebook author: Gleb Belov <<gleb@ampl.com>>

References:
1. MP solver features, IIS: https://mp.ampl.com/features-guide.html
2. Chinneck, J.W. (2008). *Feasibility and Infeasibility in Optimization: Algorithms and Computational Methods*. Springer.
3. amplpy reference, `AMPL.get_iis` and `Entity.get_values`: https://amplpy.ampl.com

In [1]:
# Install dependencies
%pip install -q amplpy pandas

In [2]:
# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=["highs"],  # modules to install
    license_uuid="default",  # license to use
)  # instantiate AMPL object and register magics

## The point

The solver says **infeasible**. Where is the conflict? An **irreducible infeasible subsystem (IIS)** answers this:
a set of constraints and variable bounds that cannot hold together, while dropping *any one* of them makes the rest
feasible. It is *irreducible* (minimal), not necessarily the *smallest* such set, and a model can have many.

With AMPL's MP solver drivers you ask for it with one option, `iis:find=1`, and read it back through a **suffix**.
This notebook shows both on a small unit commitment model, and what changes with the solver, the formulation and
AMPL's presolve.

## Suffixes in one paragraph

A **suffix** is a named value attached to a variable, constraint or objective, beside its value: `x.iis`,
`Balance[10].iis`, `Total_Cost.objpriority`. **IN** suffixes are set by you before the solve (priorities,
tolerances, relaxation prices); **OUT** suffixes are returned by the solver (IIS membership, gap and bound). The IIS
comes back in the OUT suffix `.iis` of every constraint and variable:

| `.iis` value | meaning |
|---|---|
| `non` | not in the IIS |
| `mem` | a constraint in the IIS |
| `low`, `upp` | a lower or upper bound (of a variable, or a constraint's side) in the IIS |
| `fix` | a fixed variable in the IIS |

In amplpy, `entity.get_values(["iis"])` reads the suffix for every index of an entity, and `ampl.get_iis()` returns
all members at once.

In [3]:
SOLVER = (
    "highs"  # any MP solver with IIS support: highs, gurobi, xpress, cplex, copt, ...
)

## A small unit commitment model

Each unit is on or off (`On`), produces between its minimum and its rating while on (`P`), can start and stop
(`Start`, `Stop`) with minimum up and down times and ramp limits. Every period must meet demand (`Balance`), keep
**spinning reserve** (headroom on running units, `Spinning_Reserve`) and keep enough units online for system inertia.
The cost is a defined variable, so later notebooks of this series can reuse it in a constraint.

In [4]:
MODEL = r"""
set UNITS;
param T integer > 0;
set HOURS := 1..T;
param dt > 0;                          # hours per period
param demand {HOURS} >= 0;             # MW
param reserve {HOURS} >= 0;            # spinning reserve required, MW
param min_online integer >= 0;         # units online, for system inertia
param pmin {UNITS} >= 0;
param pmax {g in UNITS} >= pmin[g];
param ramp {UNITS} > 0;                # MW per period
param minup {UNITS} integer >= 1;
param mindn {UNITS} integer >= 1;
param fuel {UNITS} >= 0;               # $/MWh
param noload {UNITS} >= 0;             # $/h while on
param startcost {UNITS} >= 0;          # $ per start
param emis {UNITS} >= 0;               # t CO2 per MWh
param on0 {UNITS} binary;              # on before the horizon
param p0 {UNITS} >= 0;                 # output before the horizon

var On {UNITS, HOURS} binary;
var Start {UNITS, HOURS} >= 0, <= 1;
var Stop {UNITS, HOURS} >= 0, <= 1;
var P {g in UNITS, HOURS} >= 0, <= pmax[g];

var Cost = sum {g in UNITS, t in HOURS}
             (dt * (fuel[g] * P[g,t] + noload[g] * On[g,t]) + startcost[g] * Start[g,t]);
minimize Total_Cost: Cost;

subject to Output_Lo {g in UNITS, t in HOURS}: P[g,t] >= pmin[g] * On[g,t];
subject to Output_Up {g in UNITS, t in HOURS}: P[g,t] <= pmax[g] * On[g,t];
subject to Balance {t in HOURS}: sum {g in UNITS} P[g,t] = demand[t];
subject to Spinning_Reserve {t in HOURS}:
   sum {g in UNITS} (pmax[g] * On[g,t] - P[g,t]) >= reserve[t];
subject to Inertia {t in HOURS}: sum {g in UNITS} On[g,t] >= min_online;
subject to Switch {g in UNITS, t in HOURS}:
   Start[g,t] - Stop[g,t] = On[g,t] - (if t = 1 then on0[g] else On[g,t-1]);
subject to Ramp_Up {g in UNITS, t in HOURS}:
   P[g,t] - (if t = 1 then p0[g] else P[g,t-1]) <= ramp[g] + pmax[g] * Start[g,t];
subject to Ramp_Down {g in UNITS, t in HOURS}:
   (if t = 1 then p0[g] else P[g,t-1]) - P[g,t] <= ramp[g] + pmax[g] * Stop[g,t];
subject to Min_Up {g in UNITS, t in HOURS}:
   sum {s in max(1, t - minup[g] + 1) .. t} Start[g,s] <= On[g,t];
subject to Min_Down {g in UNITS, t in HOURS}:
   sum {s in max(1, t - mindn[g] + 1) .. t} Stop[g,s] <= 1 - On[g,t];
"""

In [5]:
import pandas as pd

# A small thermal fleet: seven units of four types, twelve 2-hour periods
units = pd.DataFrame(
    {
        "pmin": [300, 120, 80, 20, 300, 120, 80],  # MW
        "pmax": [500, 350, 250, 100, 500, 350, 250],  # MW
        "ramp": [60, 80, 120, 100, 60, 80, 120],  # MW per period
        "minup": [12, 6, 2, 1, 12, 6, 2],  # periods
        "mindn": [12, 5, 2, 1, 12, 5, 2],  # periods
        "fuel": [9, 24, 38, 75, 9, 25, 39],  # $/MWh
        "noload": [800, 500, 300, 100, 800, 500, 300],  # $/h while on
        "startcost": [9000, 6000, 2000, 400, 9000, 6000, 2000],  # $ per start
        "emis": [0, 0.95, 0.37, 0.55, 0, 0.969, 0.3774],  # t CO2 per MWh
        "on0": [1, 1, 0, 0, 1, 1, 0],  # on before the horizon
        "p0": [400, 200, 0, 0, 400, 200, 0],  # MW before the horizon
    },
    index=["NUKE1", "COAL1", "CCGT1", "GT1", "NUKE2", "COAL2", "CCGT2"],
)
demand = [1027, 960, 1093, 1325, 1557, 1490, 1027, 960, 1557, 1656, 1457, 1093]  # MW


def load(units, reserve_frac, demand=demand, model=MODEL):
    """Reset the AMPL session, read the model and feed the data."""
    ampl.reset()
    ampl.eval(model)
    ampl.set_data(units, "UNITS")
    ampl.param["T"] = len(demand)
    ampl.param["dt"] = 2
    ampl.param["min_online"] = 2
    ampl.param["demand"] = {t + 1: d for t, d in enumerate(demand)}
    ampl.param["reserve"] = {
        t + 1: round(reserve_frac * d) for t, d in enumerate(demand)
    }


units

,pmin,pmax,ramp,minup,mindn,fuel,noload,startcost,emis,on0,p0
NUKE1,300,500,60,12,12,9,800,9000,0.0000,1,400
COAL1,120,350,80,6,5,24,500,6000,0.9500,1,200
CCGT1,80,250,120,2,2,38,300,2000,0.3700,0,0
GT1,20,100,100,1,1,75,100,400,0.5500,0,0
NUKE2,300,500,60,12,12,9,800,9000,0.0000,1,400
COAL2,120,350,80,6,5,25,500,6000,0.9690,1,200
CCGT2,80,250,120,2,2,39,300,2000,0.3774,0,0


In [6]:
load(units, reserve_frac=0.08)
ampl.solve(solver=SOLVER, verbose=False)
print(ampl.solve_result, "  cost", round(ampl.get_value("Cost")))

solved   cost 483106


## The outage: the largest unit out, and more reserve

Take `NUKE1` (500 MW) out and require reserve of 15 % of demand. The plan becomes infeasible:

In [7]:
outage = units.drop("NUKE1")
load(outage, reserve_frac=0.15)
ampl.solve(solver=SOLVER, verbose=False)
print(ampl.solve_result)

infeasible


## Ask the solver for an IIS

Solve again with `iis:find=1`, then read the IIS. `get_iis()` returns two dictionaries, variables and constraints,
each mapping a member's name to its `.iis` value:

In [8]:
ampl.solve(solver=SOLVER, **{f"{SOLVER}_options": "iis:find=1"}, verbose=False)
var_iis, con_iis = ampl.get_iis()
print("constraints:", con_iis)
print("variables:  ", var_iis)

constraints: {'Balance[10]': 'mem', 'Spinning_Reserve[10]': 'mem'}
variables:   {"On['GT1',10]": 'upp'}


The same information, one entity at a time, through the suffix itself:

In [9]:
bal = ampl.get_constraint("Balance").get_values(["iis"]).to_pandas()
bal[bal["Balance.iis"] != "non"]

,Balance.iis
10,mem


Read it as a sentence: **in period 10, demand and reserve cannot both be met, even with every unit that could help
switched on** (the upper bound of `On['GT1',10]` is the last unit the IIS needs to name). The numbers confirm it:

In [10]:
t = 10
cap = outage.pmax.sum()
need = demand[t - 1] + round(0.15 * demand[t - 1])
print(
    f"capacity {cap} MW < demand {demand[t-1]} + reserve {round(0.15*demand[t-1])} = {need} MW"
)

capacity 1800 MW < demand 1656 + reserve 248 = 1904 MW


The IIS turns "infeasible" into a short list you can act on: buy reserve elsewhere in period 10, relax the reserve
rule there, or bring capacity back. The next notebook of this series, on priced relaxation, makes that trade-off
explicit.

## The IIS depends on the solver and on the formulation

An IIS is *one* irreducible set; solvers may return different ones, and they mark members differently. The
formulation matters too: here is the same model with the on/off link written as **indicator constraints**
(`On[g,t] = 1 ==> P[g,t] >= pmin[g]`) instead of big-M rows.

In [11]:
INDICATORS = MODEL.replace(
    "subject to Output_Lo {g in UNITS, t in HOURS}: P[g,t] >= pmin[g] * On[g,t];",
    "subject to Output_Lo {g in UNITS, t in HOURS}: On[g,t] = 1 ==> P[g,t] >= pmin[g];",
).replace(
    "subject to Output_Up {g in UNITS, t in HOURS}: P[g,t] <= pmax[g] * On[g,t];",
    "subject to Output_Up {g in UNITS, t in HOURS}: On[g,t] = 0 ==> P[g,t] <= 0;",
)
for solver in [SOLVER]:  # add "gurobi", "xpress", ... if you have them
    for name, model in (("big-M rows", MODEL), ("indicators", INDICATORS)):
        load(outage, reserve_frac=0.15, model=model)
        ampl.solve(solver=solver, **{f"{solver}_options": "iis:find=1"}, verbose=False)
        var_iis, con_iis = ampl.get_iis()
        print(
            f"{solver}, {name}: {con_iis} + {len(var_iis)} bound(s): {sorted(var_iis)}"
        )

highs, big-M rows: {'Balance[10]': 'mem', 'Spinning_Reserve[10]': 'mem'} + 1 bound(s): ["On['GT1',10]"]
highs, indicators: {'Balance[10]': 'mem', 'Spinning_Reserve[10]': 'mem'} + 6 bound(s): ["On['CCGT1',10]", "On['CCGT2',10]", "On['COAL1',10]", "On['COAL2',10]", "On['GT1',10]", "On['NUKE2',10]"]


With indicators, the reserve row still needs every unit on, but the IIS now names the upper bound of `On` for every
unit: in this formulation no single unit's bound makes the rest feasible. Both answers are correct IISs of their
formulations.

Measured with other MP drivers on the same two formulations (October 2026):

| solver | big-M rows | indicators |
|---|---|---|
| HiGHS, Gurobi | `Balance[10]`, `Spinning_Reserve[10]` (`mem`) + `On['GT1',10]` (`upp`) | the same rows + the upper bound of every `On[...,10]` |
| Xpress | the same members, marked `low` (the violated side) | no rows; the binaries marked `intvar`: for a MIP the IIS may include integrality |

So read the suffix values your solver documents, and expect a different (equally valid) subsystem from another
solver or formulation.

## When AMPL's presolve finds the conflict first

AMPL simplifies the model before the solver sees it. If the conflict is simple enough, **AMPL's presolve** finds it
and the solver is never called: no solver, no `.iis` suffix. Here demand in period 10 exceeds the whole fleet's
capacity, which presolve detects from the bounds alone:

In [12]:
peak = list(demand)
peak[9] = 1850  # period 10, above the 1800 MW of the remaining fleet
load(outage, reserve_frac=0.0, demand=peak)
ampl.solve(solver=SOLVER, **{f"{SOLVER}_options": "iis:find=1"}, verbose=False)
print(ampl.solve_result, "|", ampl.get_value("solve_message"))

infeasible | presolve finds no feasible solution possible


Turn AMPL's presolve off so that the solver sees the whole model, and ask again:

In [13]:
ampl.option["presolve"] = 0
ampl.solve(solver=SOLVER, **{f"{SOLVER}_options": "iis:find=1"}, verbose=False)
var_iis, con_iis = ampl.get_iis()
print("constraints:", con_iis)
print("variables:  ", var_iis)
ampl.option["presolve"] = 10  # back to AMPL's default

constraints: {'Balance[10]': 'mem'}
variables:   {"P['COAL1',10]": 'upp', "P['CCGT1',10]": 'upp', "P['GT1',10]": 'upp', "P['NUKE2',10]": 'upp', "P['COAL2',10]": 'upp', "P['CCGT2',10]": 'upp'}


Now the IIS is the textbook capacity conflict: the balance row of period 10 and the upper bound of every unit's
output.

## Takeaways

* `iis:find=1` asks any MP solver with IIS support for an irreducible infeasible subsystem; the answer comes back in
  the OUT suffix `.iis` of every constraint and variable.
* In amplpy, `ampl.get_iis()` returns all members at once; `entity.get_values(["iis"])` reads the suffix of one
  entity, for every index.
* An IIS is minimal, not unique: solvers and formulations return different, equally valid subsystems, and mark
  members differently (`mem`, `low`, `upp`, `intvar`).
* If `solve_message` says *presolve finds no feasible solution possible*, the solver never ran: set
  `option presolve 0` and solve again to get an IIS.